In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(df_path)

print(f"Dataset shape: {df.shape}")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:

df = df.drop(['Order_ID'], axis=1)
df.head()


In [ ]:
# Task 2: Write your code here:

missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
# Define stat columns
stat_cols = ['Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']
print(f"Shape before cleaning: {df.shape}")

# Drop rows with missing stat values
df_clean = df.dropna(subset=stat_cols).copy()
print(f"Shape after cleaning: {df_clean.shape}")

In [ ]:
# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

# Fill columns with mode - discrete feature, mode is most representative
for col in ['Traffic_Level', 'Vehicle_Type']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
df_clean.head()

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:

categorical_cols = ['Weather',	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df_clean.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 6: Write your code here:

# As we have seen in the Part1 Task 5 our target distribution already so we dont want to plot the same thing again. we have seen skeweness of our data.

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1)
y = df_clean['Delivery_Time']

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True, stratify=y)

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

models = {"Random Forest Regressor": RandomForestRegressor(n_estimators=200)}

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mae': []}

In [ ]:
n_splits =5
kf = KFold(n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)

    # Store results
    all_results[mean_absolute_error]["mae"].append(mae)


In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mae']):.4f}")

In [ ]:
# Task 1: Write your code here:

feature_cols = df_clean.drop("Delivery_Time", axis=1)
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here:
from sklearn.ensemble import CatBoostRegressor

models = {"Random Forest Regressor": RandomForestRegressor(n_estimators=200),
          "CatBoost": CatBoostRegressor(verbose=0)}

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mae': []}

In [ ]:
n_splits =5
kf = KFold(n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)


    # Store results
    all_results[mean_absolute_error]["mae"].append(mae)

In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mae']):.4f}")
